# Ensemble: DINOv2 ViT-B/14 + MAE Fine-Tuned ViT-B/16 PatchCore

Max-fusion ensemble of DINOv2 ViT-B/14 block-9 with each MAE fine-tuned ViT-B/16 variant:
- **MAE 75%** - standard masking ratio (49/196 patches visible per pass)
- **MAE 25%** - lower masking ratio (147/196 patches visible per pass; stronger on Scratch)

**Key requirement:** both models must score the **same benchmark split** (metadata CSV ordering).
The original MAE runs used a different 250-defect draw from the raw pickle. This notebook
re-scores each MAE model on the benchmark split so scores align with DINOv2.

**GPU required** for re-scoring (builds memory bank queries against the saved bank).
If GPU is not available, set `DEVICE = 'cpu'` - scoring will be slow (~15-30 min).

## Expected results (indicative)

| Model | F1 | AUROC | AUPRC |
|---|---|---|---|
| Frozen ViT-B/16 + DINOv2 (max) | 0.623 | 0.967 | 0.716 |
| MAE 75% + DINOv2 (max) | TBD | TBD | TBD |
| MAE 25% + DINOv2 (max) | TBD | TBD | TBD |


## Imports


In [ ]:
import os, gc, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
from tqdm.auto import tqdm
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve,
    f1_score, precision_score, recall_score,
    precision_recall_curve, classification_report,
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_CUDA = DEVICE.type == 'cuda'
if USE_CUDA:
    torch.backends.cudnn.benchmark = True

print('Device:', DEVICE)
if USE_CUDA:
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  VRAM: {p.total_memory/1e9:.1f} GB')


## Configuration


In [ ]:
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate repo root')

# -- Benchmark split (same as DINOv2 / frozen ViT) ----------------------------
METADATA_CSV   = PROJECT_ROOT / 'data' / 'processed' / 'x224' / 'wm811k' / 'metadata_50k_5pct.csv'
TUNE_NORMAL_N  = 5_000
TEST_NORMAL_N  = 5_000
TEST_DEFECT_N  = 250
IMAGE_SIZE     = 224

# -- MAE model checkpoints -----------------------------------------------------
FT_DIR = PROJECT_ROOT / 'experiments' / 'anomaly_detection' / 'patchcore' / 'vit_b16' / 'x224' / 'FT'

MAE_CONFIGS = {
    '75pct': {
        'label'       : 'MAE fine-tuned (75% mask)',
        'ckpt'        : FT_DIR / 'MAE' / 'artifacts' / 'results' / 'best_model.pt',
        'vit_ckpt'    : FT_DIR / 'MAE' / 'artifacts' / 'checkpoints' / 'vit_finetuned.pt',
        'scores_path' : FT_DIR / 'MAE' / 'artifacts' / 'results' / 'scores_benchmark.npz',
        'out_dir'     : FT_DIR / 'MAE' / 'artifacts' / 'results' / 'ensemble_dinov2',
        'mask_ratio'  : 0.75,
    },
    '25pct': {
        'label'       : 'MAE fine-tuned (25% mask)',
        'ckpt'        : FT_DIR / 'MAE_25pct' / 'patchcore_trained_model.pt',
        'vit_ckpt'    : FT_DIR / 'MAE_25pct' / 'vit_finetuned.pt',
        'scores_path' : FT_DIR / 'MAE_25pct' / 'scores_benchmark.npz',
        'out_dir'     : FT_DIR / 'MAE_25pct' / 'ensemble_dinov2',
        'mask_ratio'  : 0.25,
    },
}

# -- DINOv2 block-9 scores (pre-computed, same benchmark split) ----------------
DINOV2_SCORES_PATH = (
    PROJECT_ROOT / 'experiments' / 'anomaly_detection' / 'patchcore'
    / 'dinov2_vit_b14' / 'x224' / 'block9' / 'artifacts' / 'results' / 'scores.npz'
)

# -- PatchCore scoring settings (must match what was used to build the bank) ---
VIT_FEATURE_BLOCK = 6
PATCH_EMBED_DIM   = 128
PATCHCORE_NN_K    = 3
TOPK_PATCH_RATIO  = 0.10
SCORE_CHUNK       = 512
BANK_BATCH        = 128
THRESHOLD_Q       = 0.95

# -- Control flags -------------------------------------------------------------
FORCE_RESCORE = False

for cfg in MAE_CONFIGS.values():
    cfg['out_dir'].mkdir(parents=True, exist_ok=True)

print(f'Project root   : {PROJECT_ROOT}')
print(f'Metadata CSV   : {METADATA_CSV}  (exists={METADATA_CSV.exists()})')
print(f'DINOv2 scores  : {DINOV2_SCORES_PATH}  (exists={DINOV2_SCORES_PATH.exists()})')
for k, cfg in MAE_CONFIGS.items():
    print(f'\nMAE {k}:')
    print(f'  ckpt      : {cfg["ckpt"]}  (exists={cfg["ckpt"].exists()})')
    print(f'  vit_ckpt  : {cfg["vit_ckpt"]}  (exists={cfg["vit_ckpt"].exists()})')
    print(f'  scores    : {cfg["scores_path"]}  (exists={cfg["scores_path"].exists()})')


## Dataset - Benchmark Split

Loads the same tune/test split used by DINOv2 (metadata CSV ordering).

**Root cause of prior F1 collapse:** the `.npy` files on disk were created with bilinear
interpolation (an older pipeline version). This produces continuous float values in `[0, 1]`
instead of clean `{0.0, 0.5, 1.0}` steps. When fed to the MAE model, normal and defect wafers
become indistinguishable in feature space (normal mean=0.739, defect mean=0.762, delta=0.023).

**Fix:** `PickleBackedWaferDataset` loads raw wafer maps from the LSWMD pickle and applies
the **exact same preprocessing as the MAE training notebook**:
`clip({0,1,2})` -> one-hot (3 channels) -> nearest-neighbor resize to 224.

The `_rebuild_export_df()` function replicates the `prepare_wm811k.py` pipeline
(same seed, same sampling order) so that `wafer_{X:07d}.npy` -> `export_df.iloc[X]`.


In [ ]:
import sys as _sys
_sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from wafer_defect.data.legacy_pickle import read_legacy_pickle, unwrap_legacy_value

RAW_PICKLE_PATH = PROJECT_ROOT / 'data' / 'raw' / 'LSWMD.pkl'


def _rebuild_export_df(raw_pickle_path: Path, seed: int = 42,
                       normal_count: int = 50_000,
                       test_defect_fraction: float = 0.05) -> pd.DataFrame:
    """Reproduces the export_df created by prepare_wm811k.py (ignore_index=True concat).
    wafer_{X:07d}.npy  <->  export_df.iloc[X]['waferMap'].
    """
    print('Loading raw pickle (~30s)..')
    df = read_legacy_pickle(str(raw_pickle_path))

    def _infer_label(row):
        failure = unwrap_legacy_value(row.get('failureType', '')).lower()
        if failure and failure != 'none':
            return 'pattern'
        if failure == 'none':
            return 'none'
        return None

    df = df.copy()
    df['label'] = df.apply(_infer_label, axis=1)
    df = df[df['label'].notna()].copy()

    normal_df = df[df['label'] == 'none'].copy()
    defect_df = df[df['label'] == 'pattern'].copy()
    print(f'Labeled: {len(df):,}  Normal: {len(normal_df):,}  Defect: {len(defect_df):,}')

    # Sample normals - same call as prepare_wm811k.py
    normal_df = normal_df.sample(n=min(normal_count, len(normal_df)), random_state=seed)

    # split_normals: shuffle then 80/10/10
    normal_df = normal_df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    n = len(normal_df)
    train_end, val_end = int(0.8 * n), int(0.9 * n)
    normal_df.loc[:train_end - 1,       'split'] = 'train'
    normal_df.loc[train_end:val_end - 1, 'split'] = 'val'
    normal_df.loc[val_end:,              'split'] = 'test'

    # sample_test_defects
    test_normal_count = int((normal_df['split'] == 'test').sum())
    requested = max(1, int(round(test_normal_count * test_defect_fraction)))
    defect_df = defect_df.sample(n=min(requested, len(defect_df)), random_state=seed).copy()
    defect_df['split'] = 'test'

    export_df = pd.concat([normal_df, defect_df], ignore_index=True)
    print(f'export_df: {len(export_df):,} rows  '
          f'(train={int((export_df["split"]=="train").sum()):,}  '
          f'val={int((export_df["split"]=="val").sum()):,}  '
          f'test={int((export_df["split"]=="test").sum()):,})')
    return export_df


class PickleBackedWaferDataset(Dataset):
    """Loads raw {0,1,2} wafer maps from the pickle-derived export_df.

    Uses the filename index embedded in the metadata CSV array_path to look up
    export_df.iloc[X]['waferMap'], then applies the exact same preprocessing
    as the MAE training notebook:
        clip({0,1,2})  ->  one-hot(3 channels)  ->  nearest-neighbor resize to 224
    """
    def __init__(self, meta_df: pd.DataFrame, export_df: pd.DataFrame, size: int = 224):
        self.indices  = [
            int(Path(str(p)).stem.replace('wafer_', ''))
            for p in meta_df['array_path']
        ]
        self.raw_maps = export_df['waferMap'].values
        self.size     = size

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        row_idx = self.indices[idx]
        arr = np.clip(np.array(self.raw_maps[row_idx], dtype=np.int64), 0, 2)
        x   = torch.tensor(arr, dtype=torch.long)
        x   = F.one_hot(x, num_classes=3).permute(2, 0, 1).float()   # [3, H, W]
        x   = F.interpolate(
                  x.unsqueeze(0), size=(self.size, self.size), mode='nearest'
              ).squeeze(0)
        return x, 0


# -- Build export_df (maps filename index -> raw wafer map) --------------------
export_df = _rebuild_export_df(RAW_PICKLE_PATH)

# -- Load benchmark split from metadata CSV ------------------------------------
meta = pd.read_csv(METADATA_CSV)

tune_df        = meta[(meta['split'] == 'val')  & (meta['is_anomaly'] == 0)].head(TUNE_NORMAL_N)
test_normal_df = meta[(meta['split'] == 'test') & (meta['is_anomaly'] == 0)].head(TEST_NORMAL_N)
test_defect_df = meta[(meta['split'] == 'test') & (meta['is_anomaly'] == 1)].head(TEST_DEFECT_N)

defect_class_labels = test_defect_df['defect_type'].values

print(f'\nTune normal  : {len(tune_df):,}')
print(f'Test normal  : {len(test_normal_df):,}')
print(f'Test defect  : {len(test_defect_df):,}')
print('\nDefect class counts (benchmark split):')
print(pd.Series(defect_class_labels).value_counts().to_string())

# Smoke-test: verify index lookup gives correct wafer dimensions
_sample_idx = int(Path(str(tune_df['array_path'].iloc[0])).stem.replace('wafer_', ''))
_sample_map = np.array(export_df.iloc[_sample_idx]['waferMap'], dtype=np.int64)
print(f'\nSmoke-test: index={_sample_idx}  shape={_sample_map.shape}  '
      f'unique={np.unique(_sample_map)}')


## Model Architecture

Defines the `FineTunedPatchExtractor` used by the MAE experiments.
This must exactly match the architecture used to build the saved memory bank.


In [ ]:
class FineTunedPatchExtractor(nn.Module):
    """Hooks block[VIT_FEATURE_BLOCK] for spatial patch tokens."""

    def __init__(self, vit: nn.Module, block_idx: int = VIT_FEATURE_BLOCK,
                 proj_dim: int = PATCH_EMBED_DIM):
        super().__init__()
        self.vit   = vit
        self._feat = None
        self.vit.blocks[block_idx].register_forward_hook(
            lambda m, i, o: setattr(self, '_feat', o)
        )
        self.proj = nn.Linear(vit.embed_dim, proj_dim, bias=False)

    def forward(self, x):
        self.vit(x)
        return self._feat[:, 1:, :]  # drop CLS -> [B, 196, 768]


def load_mae_extractor(cfg: dict, device: torch.device) -> tuple:
    """Load fine-tuned ViT backbone and memory bank from checkpoint.
    Returns (extractor, memory_bank) where memory_bank may be None.
    """
    ckpt_path = cfg['ckpt']
    vit_path  = cfg['vit_ckpt']

    # Build base ViT
    base_vit = timm.create_model(
        'vit_base_patch16_224.augreg_in21k_ft_in1k',
        pretrained=False,
        num_classes=0,
    )
    extractor = FineTunedPatchExtractor(base_vit).to(device)

    memory_bank = None

    if ckpt_path.exists():
        print(f'  Loading full checkpoint: {ckpt_path.name}')
        ckpt = torch.load(str(ckpt_path), map_location='cpu')

        if isinstance(ckpt, dict):
            # Try to load extractor state
            if 'extractor_state' in ckpt:
                extractor.load_state_dict(ckpt['extractor_state'])
                print('  extractor_state loaded')
            elif 'vit_state_dict' in ckpt:
                extractor.vit.load_state_dict(ckpt['vit_state_dict'])
                print('  vit_state_dict loaded (projection layer is random - rebuilding bank needed)')

            # Try to load memory bank
            if 'memory_bank' in ckpt:
                memory_bank = ckpt['memory_bank'].to(device)
                print(f'  memory_bank loaded: {memory_bank.shape}')
            else:
                print('  memory_bank NOT in checkpoint')
        else:
            print('  Checkpoint is not a dict - cannot load')

    elif vit_path.exists():
        print(f'  Full checkpoint missing. Loading ViT backbone from: {vit_path.name}')
        ckpt = torch.load(str(vit_path), map_location='cpu')
        if isinstance(ckpt, dict) and 'vit_state_dict' in ckpt:
            extractor.vit.load_state_dict(ckpt['vit_state_dict'])
        else:
            extractor.vit.load_state_dict(ckpt)
        print('  ViT backbone loaded (projection layer is random - memory bank must be rebuilt)')
    else:
        raise FileNotFoundError(
            f'No checkpoint found at {ckpt_path} or {vit_path}'
        )

    extractor.eval()
    for p in extractor.parameters():
        p.requires_grad = False

    return extractor, memory_bank


print('Model class defined.')


## Scoring Helpers


In [ ]:
def extract_patch_embeddings(extractor, xb, device):
    """L2-normalised patch embeddings: [B*196, proj_dim]."""
    with torch.inference_mode():
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_CUDA):
            feat = extractor(xb)
            emb  = extractor.proj(feat)
        emb = emb.float().reshape(-1, emb.shape[-1])
        emb = F.normalize(emb, p=2, dim=1)
    return emb


def score_split(extractor, memory_bank, loader, desc='Scoring'):
    """Score all images in loader against the memory bank."""
    scores = []
    for xb, _ in tqdm(loader, desc=desc, leave=False):
        xb  = xb.to(DEVICE)
        emb = extract_patch_embeddings(extractor, xb, DEVICE)
        B   = len(xb)
        P   = emb.shape[0] // B

        patch_dists = torch.empty(B * P, dtype=torch.float32, device=DEVICE)
        for start in range(0, B * P, SCORE_CHUNK):
            end   = min(start + SCORE_CHUNK, B * P)
            chunk = emb[start:end]
            dists = torch.cdist(chunk, memory_bank)
            knn   = dists.topk(PATCHCORE_NN_K, dim=1, largest=False).values
            patch_dists[start:end] = knn.mean(dim=1)

        patch_dists = patch_dists.view(B, P)
        topk        = max(1, int(P * TOPK_PATCH_RATIO))
        img_scores  = patch_dists.topk(topk, dim=1).values.mean(dim=1)
        scores.extend(img_scores.cpu().tolist())
    return np.array(scores)


def build_bank_from_train(extractor, train_df, export_df, memory_bank_max=600_000):
    """Build PatchCore memory bank from training normals using raw pickle data."""
    print('  Building memory bank from training normals..')
    train_ds = PickleBackedWaferDataset(train_df, export_df, IMAGE_SIZE)
    loader   = DataLoader(train_ds, batch_size=BANK_BATCH, shuffle=False,
                          num_workers=0, pin_memory=USE_CUDA)
    sampled, total_seen, sample_ratio = [], 0, None

    for step, (xb, _) in enumerate(tqdm(loader, desc='  Bank build', leave=False)):
        xb  = xb.to(DEVICE)
        emb = extract_patch_embeddings(extractor, xb, DEVICE)
        total_seen += len(emb)
        if sample_ratio is None:
            tokens_per_img  = len(emb) // len(xb)
            estimated_total = tokens_per_img * len(train_ds)
            sample_ratio    = min(1.0, memory_bank_max / estimated_total)
        if sample_ratio < 1.0:
            k   = max(1, int(round(len(emb) * sample_ratio)))
            idx = torch.randperm(len(emb), device=DEVICE)[:k]
            emb = emb[idx]
        sampled.append(emb)

    bank = torch.cat(sampled, dim=0)
    if len(bank) > memory_bank_max:
        idx = torch.randperm(len(bank))[:memory_bank_max]
        bank = bank[idx]
    print(f'  Bank size: {len(bank):,} x {bank.shape[1]}')
    return bank


print('Scoring helpers defined.')


## Generate / Load MAE Scores

For each MAE variant:
1. Load `scores_benchmark.npz` if it exists and `FORCE_RESCORE=False`
2. Otherwise, load checkpoint (extractor + memory bank), score the benchmark split, save npz


In [ ]:
loader_kw = dict(batch_size=BANK_BATCH, shuffle=False, num_workers=0, pin_memory=USE_CUDA)
tune_loader   = DataLoader(PickleBackedWaferDataset(tune_df,        export_df, IMAGE_SIZE), **loader_kw)
normal_loader = DataLoader(PickleBackedWaferDataset(test_normal_df, export_df, IMAGE_SIZE), **loader_kw)
defect_loader = DataLoader(PickleBackedWaferDataset(test_defect_df, export_df, IMAGE_SIZE), **loader_kw)

mae_scores = {}  # variant -> dict(tune, normal, defect)

for variant, cfg in MAE_CONFIGS.items():
    print(f'\n-- {cfg["label"]} ------------------------------')
    scores_path = cfg['scores_path']

    if scores_path.exists() and not FORCE_RESCORE:
        print(f'Loading saved scores: {scores_path.name}')
        with np.load(str(scores_path)) as f:
            mae_scores[variant] = {
                'tune'  : f['tune_normal_scores'],
                'normal': f['test_normal_scores'],
                'defect': f['test_defect_scores'],
            }
        s = mae_scores[variant]
        print(f'  tune  : mean={s["tune"].mean():.4f}  std={s["tune"].std():.4f}')
        print(f'  normal: mean={s["normal"].mean():.4f}  std={s["normal"].std():.4f}')
        print(f'  defect: mean={s["defect"].mean():.4f}  std={s["defect"].std():.4f}')
        print(f'  defect-normal delta: {s["defect"].mean()-s["normal"].mean():.4f}')
        continue

    # Load model
    extractor, memory_bank = load_mae_extractor(cfg, DEVICE)

    # If memory bank not in checkpoint, rebuild from training normals
    if memory_bank is None:
        print('  Memory bank missing - rebuilding from train normals')
        train_df_mae = meta[(meta['split'] == 'train') & (meta['is_anomaly'] == 0)]
        memory_bank  = build_bank_from_train(extractor, train_df_mae, export_df)

    # Score benchmark splits
    print('  Scoring tune normals..')
    tune_s   = score_split(extractor, memory_bank, tune_loader,   'Tune-normal')
    print('  Scoring test normals..')
    normal_s = score_split(extractor, memory_bank, normal_loader, 'Test-normal')
    print('  Scoring test defects..')
    defect_s = score_split(extractor, memory_bank, defect_loader, 'Test-defect')

    mae_scores[variant] = {'tune': tune_s, 'normal': normal_s, 'defect': defect_s}

    print(f'  tune  : mean={tune_s.mean():.4f}  std={tune_s.std():.4f}')
    print(f'  normal: mean={normal_s.mean():.4f}  std={normal_s.std():.4f}')
    print(f'  defect: mean={defect_s.mean():.4f}  std={defect_s.std():.4f}')
    print(f'  defect-normal delta: {defect_s.mean()-normal_s.mean():.4f}')

    np.savez_compressed(
        str(scores_path),
        tune_normal_scores=tune_s,
        test_normal_scores=normal_s,
        test_defect_scores=defect_s,
    )
    print(f'  Scores saved -> {scores_path}')

    del extractor, memory_bank
    gc.collect()
    if USE_CUDA:
        torch.cuda.empty_cache()

print('\nAll MAE scores ready.')


## Load DINOv2 Scores


In [ ]:
with np.load(str(DINOV2_SCORES_PATH)) as f:
    d2_tune   = f['tune_normal_scores'].astype(np.float64)
    d2_normal = f['test_normal_scores'].astype(np.float64)
    d2_defect = f['test_defect_scores'].astype(np.float64)

assert len(d2_tune)   == TUNE_NORMAL_N
assert len(d2_normal) == TEST_NORMAL_N
assert len(d2_defect) == TEST_DEFECT_N
print(f'DINOv2 block-9 scores loaded. Defect: mean={d2_defect.mean():.4f}')


## Ensemble: Max-Fusion for Each Variant

Both model score sets are min-max normalised to [0, 1] independently before max-fusion.
Threshold is calibrated at q=0.95 on the ensemble tune-normal scores.


In [ ]:
def minmax_norm(tune, normal, defect):
    all_s = np.concatenate([tune, normal, defect])
    lo, hi = all_s.min(), all_s.max()
    s = hi - lo
    return (tune-lo)/s, (normal-lo)/s, (defect-lo)/s


def evaluate_ensemble(mae_tune, mae_normal, mae_defect,
                      d2_tune, d2_normal, d2_defect,
                      defect_class_labels, threshold_q=THRESHOLD_Q):
    """Normalise, max-fuse, threshold, evaluate. Returns metrics dict."""
    mae_t, mae_n, mae_d = minmax_norm(mae_tune,  mae_normal,  mae_defect)
    d2_t,  d2_n,  d2_d  = minmax_norm(d2_tune,   d2_normal,   d2_defect)

    ens_tune   = np.maximum(mae_t, d2_t)
    ens_normal = np.maximum(mae_n, d2_n)
    ens_defect = np.maximum(mae_d, d2_d)

    thresh = float(np.quantile(ens_tune, threshold_q))

    all_scores = np.concatenate([ens_normal, ens_defect])
    all_labels = np.concatenate([
        np.zeros(len(ens_normal), dtype=int),
        np.ones(len(ens_defect),  dtype=int),
    ])
    preds = (all_scores > thresh).astype(int)

    auroc = roc_auc_score(all_labels, all_scores)
    auprc = average_precision_score(all_labels, all_scores)
    f1    = f1_score(all_labels, preds, pos_label=1, zero_division=0)
    prec  = precision_score(all_labels, preds, pos_label=1, zero_division=0)
    rec   = recall_score(all_labels, preds, pos_label=1, zero_division=0)

    defect_preds = (ens_defect > thresh).astype(int)
    per_class = {}
    for cls in sorted(np.unique(defect_class_labels)):
        mask = defect_class_labels == cls
        per_class[cls] = {
            'n': int(mask.sum()),
            'detected': int(defect_preds[mask].sum()),
            'recall': float(defect_preds[mask].sum() / mask.sum()),
        }

    return {
        'auroc': auroc, 'auprc': auprc, 'f1': f1,
        'precision': prec, 'recall': rec,
        'threshold': thresh,
        'per_class': per_class,
        'ens_tune': ens_tune, 'ens_normal': ens_normal, 'ens_defect': ens_defect,
        'all_scores': all_scores, 'all_labels': all_labels, 'preds': preds,
    }


results = {}
for variant, cfg in MAE_CONFIGS.items():
    s = mae_scores[variant]
    results[variant] = evaluate_ensemble(
        s['tune'].astype(np.float64), s['normal'].astype(np.float64), s['defect'].astype(np.float64),
        d2_tune, d2_normal, d2_defect,
        defect_class_labels,
    )
    r = results[variant]
    print(f"\n-- {cfg['label']} + DINOv2 (max-fusion) --")
    print(f"  F1={r['f1']:.4f}  AUROC={r['auroc']:.4f}  AUPRC={r['auprc']:.4f}")
    print(f"  Recall={r['recall']:.4f}  Precision={r['precision']:.4f}")
    print('  Per-class recall:')
    for cls, v in r['per_class'].items():
        print(f'    {cls:<12}: {v["recall"]:.3f}  (n={v["n"]})')


## Comparison Table


In [ ]:
all_classes = sorted(np.unique(defect_class_labels))

rows = []
# Reference: frozen ViT + DINOv2 (from saved report numbers)
rows.append({
    'Model': 'Frozen ViT-B/16 + DINOv2 (max)',
    'F1': 0.623, 'AUROC': 0.967, 'AUPRC': 0.716,
    'Center': 0.880, 'Donut': 1.000, 'Edge-Loc': 0.849,
    'Edge-Ring': 0.893, 'Loc': 0.912, 'Near-full': 1.000,
    'Random': 1.000, 'Scratch': 0.800,
})

for variant, cfg in MAE_CONFIGS.items():
    r = results[variant]
    row = {
        'Model': f"{cfg['label']} + DINOv2 (max)",
        'F1': round(r['f1'], 4),
        'AUROC': round(r['auroc'], 4),
        'AUPRC': round(r['auprc'], 4),
    }
    for cls in all_classes:
        row[cls] = round(r['per_class'].get(cls, {}).get('recall', float('nan')), 3)
    rows.append(row)

compare_df = pd.DataFrame(rows).set_index('Model')
print('\n-- Ensemble Comparison ----------------------------------------------')
display(compare_df.round(4))

# Save
for variant, cfg in MAE_CONFIGS.items():
    compare_df.to_csv(str(cfg['out_dir'] / 'ensemble_comparison.csv'))
    with open(str(cfg['out_dir'] / 'evaluation_metrics.json'), 'w') as fh:
        r = results[variant]
        json.dump({
            'ensemble': f"{cfg['label']} + DINOv2 block-9 (max-fusion)",
            'roc_auc': r['auroc'], 'auprc': r['auprc'], 'f1': r['f1'],
            'precision': r['precision'], 'recall': r['recall'],
            'threshold': r['threshold'],
            'per_class': {k: {kk: vv for kk, vv in v.items() if kk != 'mean_score'}
                          for k, v in r['per_class'].items()},
        }, fh, indent=2)
print('Metrics saved.')


## Plots: Score Distributions and Per-Class Recall


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
variant_list = list(MAE_CONFIGS.keys())
colors = ['steelblue', 'darkorange']

for col, (variant, cfg) in enumerate(MAE_CONFIGS.items()):
    r = results[variant]
    ax_dist = axes[0, col]
    ax_dist.hist(r['ens_normal'], bins=50, alpha=0.7, color='steelblue', label='Normal')
    ax_dist.hist(r['ens_defect'], bins=50, alpha=0.7, color='tomato',    label='Defect')
    ax_dist.axvline(r['threshold'], color='black', linestyle='--', lw=2,
                    label=f"={r['threshold']:.4f}")
    ax_dist.set_title(f"{cfg['label']}\n+ DINOv2  F1={r['f1']:.4f} AUROC={r['auroc']:.4f}")
    ax_dist.set_xlabel('Ensemble Score'); ax_dist.set_ylabel('Count')
    ax_dist.legend(fontsize=8)

# Per-class recall grouped bars
ax_pc = axes[0, 2]
x     = np.arange(len(all_classes))
width = 0.35

# Reference bars (frozen ViT + DINOv2)
ref_row = compare_df.loc['Frozen ViT-B/16 + DINOv2 (max)']

for col, (variant, cfg) in enumerate(MAE_CONFIGS.items()):
    r = results[variant]
    recalls = [r['per_class'].get(c, {}).get('recall', 0) for c in all_classes]
    ax_pc.bar(x + (col - 0.5) * width, recalls, width * 0.9,
              label=cfg['label'][:8] + '.', color=colors[col], alpha=0.85)

ax_pc.set_xticks(x); ax_pc.set_xticklabels(all_classes, rotation=30, ha='right', fontsize=8)
ax_pc.set_ylim(0, 1.15); ax_pc.set_ylabel('Recall')
ax_pc.set_title('Per-class recall: MAE 75% vs 25% ensembles')
ax_pc.legend(fontsize=8)

# ROC comparison
ax_roc = axes[1, 0]
for (variant, cfg), color in zip(MAE_CONFIGS.items(), colors):
    r = results[variant]
    fpr, tpr, _ = roc_curve(r['all_labels'], r['all_scores'])
    ax_roc.plot(fpr, tpr, color=color, lw=2,
                label=f"{cfg['label'][:8]}. AUC={r['auroc']:.4f}")
ax_roc.plot([0,1],[0,1],'k--',alpha=0.4)
ax_roc.set_title('ROC Curves'); ax_roc.set_xlabel('FPR'); ax_roc.set_ylabel('TPR')
ax_roc.legend(fontsize=8)

# PR comparison
ax_pr = axes[1, 1]
for (variant, cfg), color in zip(MAE_CONFIGS.items(), colors):
    r = results[variant]
    p_arr, rec_arr, _ = precision_recall_curve(r['all_labels'], r['all_scores'])
    ax_pr.plot(rec_arr, p_arr, color=color, lw=2,
               label=f"{cfg['label'][:8]}. AUPRC={r['auprc']:.4f}")
baseline = TEST_DEFECT_N / (TEST_DEFECT_N + TEST_NORMAL_N)
ax_pr.axhline(baseline, color='gray', linestyle='--', lw=1)
ax_pr.set_title('PR Curves'); ax_pr.set_xlabel('Recall'); ax_pr.set_ylabel('Precision')
ax_pr.set_xlim(0,1); ax_pr.set_ylim(0,1.02); ax_pr.legend(fontsize=8)

# Summary bar (F1 / AUROC / AUPRC)
ax_sum = axes[1, 2]
metrics_to_show = ['F1', 'AUROC', 'AUPRC']
labels_show = ['Frozen+DINOv2'] + [f"MAE {v}+DINOv2" for v in variant_list]
vals = compare_df[metrics_to_show].values
x2 = np.arange(len(metrics_to_show))
w2 = 0.25
bar_colors_sum = ['gray', 'steelblue', 'darkorange']
for i, (lbl, row, bc) in enumerate(zip(labels_show, vals, bar_colors_sum)):
    ax_sum.bar(x2 + (i - 1) * w2, row, w2 * 0.9, label=lbl, color=bc, alpha=0.85)
ax_sum.set_xticks(x2); ax_sum.set_xticklabels(metrics_to_show)
ax_sum.set_ylim(0.5, 1.05); ax_sum.set_ylabel('Score')
ax_sum.set_title('Headline metrics comparison')
ax_sum.legend(fontsize=7)

plt.suptitle('DINOv2 ViT-B/14 + MAE ViT-B/16 Ensemble Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()

fig_path = FT_DIR / 'mae_dinov2_ensemble_comparison.png'
plt.savefig(str(fig_path), dpi=130, bbox_inches='tight')
plt.show()
print(f'Figure saved -> {fig_path}')


## Save Outputs


In [ ]:
# Standardized local artifact export

ARTIFACT_DIR = FT_DIR / 'artifacts' / 'mae_dinov2_ensemble'
CHECKPOINTS_DIR = ARTIFACT_DIR / 'checkpoints'
PLOTS_DIR = ARTIFACT_DIR / 'plots'
RESULTS_DIR = ARTIFACT_DIR / 'results'
for _path in [CHECKPOINTS_DIR, PLOTS_DIR, RESULTS_DIR]:
    _path.mkdir(parents=True, exist_ok=True)

best_variant = max(results, key=lambda key: (results[key]['f1'], results[key]['auroc'], results[key]['auprc']))
best_cfg = MAE_CONFIGS[best_variant]
best_result = results[best_variant]

mae_tune = mae_scores[best_variant]['tune']
mae_normal = mae_scores[best_variant]['normal']
mae_defect = mae_scores[best_variant]['defect']
mae_t, mae_n, mae_d = minmax_norm(mae_tune, mae_normal, mae_defect)
d2_t, d2_n, d2_d = minmax_norm(d2_tune, d2_normal, d2_defect)

test_scores_df = pd.concat([
    test_normal_df[['failure_label', 'is_anomaly']].assign(split='test_normal'),
    test_defect_df[['failure_label', 'is_anomaly']].assign(split='test_defect'),
], ignore_index=True)
test_scores_df['score'] = np.concatenate([best_result['ens_normal'], best_result['ens_defect']])
test_scores_df['predicted_anomaly'] = (test_scores_df['score'] > float(best_result['threshold'])).astype(int)
test_scores_df.to_csv(RESULTS_DIR / 'test_scores.csv', index=False)

defect_recall_df = pd.DataFrame([
    {'failure_label': label, **stats}
    for label, stats in sorted(best_result['per_class'].items())
]).rename(columns={'n': 'count'})
defect_recall_df.to_csv(RESULTS_DIR / 'test_defect_recall.csv', index=False)

feature_rows = np.column_stack([
    np.concatenate([mae_n, mae_d]),
    np.concatenate([d2_n, d2_d]),
    np.concatenate([best_result['ens_normal'], best_result['ens_defect']]),
])
umap_df = test_scores_df[['failure_label', 'is_anomaly', 'split', 'score', 'predicted_anomaly']].copy()
umap_df.insert(0, 'point_index', np.arange(len(umap_df)))

try:
    import umap.umap_ as umap
except Exception:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'umap-learn', '-q'])
    import umap.umap_ as umap

coords = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric='euclidean', random_state=42, transform_seed=42, low_memory=True).fit_transform(feature_rows)
umap_df['umap_1'] = coords[:, 0]
umap_df['umap_2'] = coords[:, 1]
umap_df.to_csv(RESULTS_DIR / 'umap_test_embeddings.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 6))
normal_mask = umap_df['is_anomaly'].to_numpy() == 0
ax.scatter(coords[normal_mask, 0], coords[normal_mask, 1], s=8, alpha=0.45, c='steelblue', label='Normal', linewidths=0)
ax.scatter(coords[~normal_mask, 0], coords[~normal_mask, 1], s=8, alpha=0.60, c='tomato', label='Defect', linewidths=0)
ax.set_title('UMAP of Ensemble Test Scores')
ax.set_xlabel('UMAP-1')
ax.set_ylabel('UMAP-2')
ax.legend()
fig.tight_layout()
fig.savefig(PLOTS_DIR / 'umap_test_embeddings.png', dpi=160, bbox_inches='tight')
plt.show()
plt.close(fig)

artifact = {
    'selected_variant': best_variant,
    'label': best_cfg['label'],
    'threshold_z': float(best_result['threshold']),
    'source_scores_path': str(best_cfg['scores_path']),
    'source_mae_checkpoint': str(best_cfg['ckpt']),
    'source_dinov2_scores_path': str(DINOV2_SCORES_PATH),
}
torch.save(artifact, CHECKPOINTS_DIR / 'model.pt')
(RESULTS_DIR / 'evaluation_metrics.json').write_text(json.dumps(artifact | {
    'f1': float(best_result['f1']),
    'auroc': float(best_result['auroc']),
    'auprc': float(best_result['auprc']),
}, indent=2), encoding='utf-8')
